# 🔍 DefectEye — Industrial Anomaly Detection (PaDiM)

Train an unsupervised defect detector on **defect-free images only**, then localize unseen defects.

## Before you Run All
1. **Add Data** (right panel) → search **`MVTec AD`** → add the dataset.
2. **Settings → Accelerator → GPU** (P100 / T4).
3. Edit `REPO_URL` below to your GitHub fork, then **Run All**.

Everything runs in the cloud — nothing is stored on your laptop.

In [ ]:
# 1) Get the code + install the few extra deps (torch/torchvision already on Kaggle)
REPO_URL = "https://github.com/TahaMazhar01/DefectEye-Computer-Vision.git"

import os
REPO_DIR = os.path.splitext(os.path.basename(REPO_URL))[0]
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL
%cd $REPO_DIR
!pip install -q scipy scikit-learn opencv-python-headless gradio tqdm

In [ ]:
# 2) Find the dataset path. MVTec category folders should sit directly under DATA_ROOT.
!ls /kaggle/input
print('---')
# Most MVTec Kaggle mirrors extract to one of these. Adjust if your ls output differs.
import os
candidates = [
    '/kaggle/input/mvtec-ad',
    '/kaggle/input/mvtec-ad-dataset',
    '/kaggle/input/mvtecad',
]
DATA_ROOT = next((c for c in candidates if os.path.isdir(c)), '/kaggle/input')
print('DATA_ROOT =', DATA_ROOT)
print('categories:', sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))))

In [ ]:
# 3) Config
CATEGORY = 'bottle'              # pick any category printed above
BACKBONE = 'wide_resnet50_2'    # 'resnet18' (fast) or 'wide_resnet50_2' (best)

In [ ]:
# 4) Train (fits per-patch Gaussians on defect-free images)
!python -m src.train --data_root "{DATA_ROOT}" --category {CATEGORY} --backbone {BACKBONE}

In [ ]:
# 5) Evaluate (image + pixel AUROC, saves a sample grid)
!python -m src.evaluate --data_root "{DATA_ROOT}" --category {CATEGORY} --backbone {BACKBONE} \
    --model models/{CATEGORY}_{BACKBONE}.pt

In [ ]:
# 6) Show qualitative results: input | prediction | ground truth
from IPython.display import Image as IPImage
IPImage(filename=f'results/{CATEGORY}_{BACKBONE}_samples.png')

In [ ]:
# 7) (Optional) Launch a temporary public demo link straight from Kaggle
# !python -c "import app; app.demo.launch(share=True)"

## Next steps
- Loop over several categories to fill the results table in the README.
- Download a trained `models/*.pt` and deploy `app.py` on **Hugging Face Spaces** for a permanent live link.
- Record a short screen capture of the demo for your portfolio.